# LeetCode 1898 — Maximum Number of Removable Characters bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-removable-characters/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, String, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def maximumRemovals(self, s: str, p: str, removable: List[int]) -> int:
        ...
```

Lộ trình:
1. **Thử `k = 0, 1, 2, …`** và kiểm tra `p` còn là dãy con không — dừng ở `k` đầu tiên thất bại (`O(r · n)`).
2. **Tìm kiếm nhị phân trên `k`** vì tính chất "còn là dãy con" **đơn điệu** theo `k` — mỗi lần kiểm tra dựng tập chỉ số bị xóa (`O(n log r)`).
3. **Mảng "thời điểm bị xóa"** `when[i]` = vị trí của `i` trong `removable` (∞ nếu không bao giờ xóa): ký tự `i` còn sống sau `k` lần xóa ⇔ `when[i] ≥ k` — kiểm tra không cần dựng tập.
4. Bonus: kiểm tra dãy con bằng **iterator** (`all(c in it for c in p)`) và so sánh hằng số thời gian.

## 1. Mục tiêu học

1. Kiểm tra **dãy con (subsequence)** bằng hai con trỏ trong `O(|s|)`.
2. Nhận ra tính **đơn điệu**: xóa thêm ký tự không thể làm `p` "trở lại" thành dãy con ⇒ nhị phân trên `k`.
3. Cài khuôn nhị phân **"tìm `k` lớn nhất thỏa"** với `mid` làm tròn lên.
4. Dùng mảng **thời điểm xóa** để mỗi lần kiểm tra không cần tạo `set` mới — tối ưu hằng số và bộ nhớ.
5. Kiểm chứng bằng cách thử tuyến tính mọi `k` trên chuỗi ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `p` với `p` là **dãy con** của `s`, và mảng `removable` gồm các chỉ số **phân biệt** của `s`.

Chọn số nguyên `k` (`0 ≤ k ≤ removable.length`) sao cho sau khi **xóa** `k` ký tự ở các chỉ số `removable[0], …, removable[k−1]` khỏi `s`, `p` **vẫn** là dãy con. (Chính xác: đánh dấu các ký tự `s[removable[i]]` với `i < k`, xóa mọi ký tự đã đánh dấu, rồi kiểm tra.)

Trả về `k` **lớn nhất** có thể.

**Ví dụ 1:** `s = "abcacb"`, `p = "ab"`, `removable = [3,1,0]` → `2`. Xóa chỉ số 3 và 1: `"accb"` — `"ab"` vẫn là dãy con. Xóa thêm 0: `"ccb"` — không còn.

**Ví dụ 2:** `s = "abcbddddd"`, `p = "abcd"`, `removable = [3,2,1,4,5,6]` → `1`.

**Ví dụ 3:** `s = "abcab"`, `p = "abc"`, `removable = [0,1,2,3,4]` → `0`.

**Ràng buộc:** `1 ≤ |p| ≤ |s| ≤ 10⁵`, `0 ≤ removable.length < |s|`, chỉ số phân biệt, `p` là dãy con của `s`, chữ thường.

In [1]:
import random
import time
from typing import List

examples = [
    (("abcacb", "ab", [3, 1, 0]), 2),
    (("abcbddddd", "abcd", [3, 2, 1, 4, 5, 6]), 1),
    (("abcab", "abc", [0, 1, 2, 3, 4]), 0),
]
for (s, p, rem), want in examples:
    print(f"s={s!r}, p={p!r}, removable={rem} -> kỳ vọng {want}")

s='abcacb', p='ab', removable=[3, 1, 0] -> kỳ vọng 2
s='abcbddddd', p='abcd', removable=[3, 2, 1, 4, 5, 6] -> kỳ vọng 1
s='abcab', p='abc', removable=[0, 1, 2, 3, 4] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Kiểm tra dãy con bằng hai con trỏ

Duyệt `s`; con trỏ `j` trên `p` tiến khi `s[i] == p[j]`. `p` là dãy con ⇔ cuối cùng `j == len(p)`. Tham lam "khớp sớm nhất" là tối ưu.

In [2]:
def is_subsequence(p, s):
    j = 0
    for ch in s:
        if j < len(p) and ch == p[j]:
            j += 1
    return j == len(p)

print(is_subsequence("ab", "accb"), is_subsequence("ab", "ccb"), is_subsequence("", "xyz"))

True False True


### 3.2 Kiểm tra dãy con bằng iterator

`it = iter(s)`; `c in it` tiêu thụ iterator tới khi gặp `c`. `all(c in it for c in p)` — gọn và chạy nhanh (vòng lặp trong C).

In [3]:
def is_subseq_iter(p, s):
    it = iter(s)
    return all(c in it for c in p)

print(is_subseq_iter("ab", "accb"), is_subseq_iter("ab", "ccb"))

True False


### 3.3 Xóa theo tập chỉ số

Chuỗi sau khi xóa: `"".join(ch for i, ch in enumerate(s) if i not in removed)`.

In [4]:
s = "abcacb"; removed = {3, 1}
print("".join(ch for i, ch in enumerate(s) if i not in removed))

accb


### 3.4 Khuôn nhị phân "lớn nhất thỏa"

`lo` luôn thỏa (ở đây `k = 0` luôn thỏa vì đề bảo đảm `p` là dãy con của `s`), `hi` là cận trên; `mid = (lo + hi + 1) // 2`.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `s = "abcacb"`, `p = "ab"`, `removable = [3, 1, 0]`:

| k | chỉ số đã xóa | chuỗi còn lại | "ab" là dãy con? |
|---|---|---|---|
| 0 | {} | abcacb | có |
| 1 | {3} | abccb | có |
| 2 | {3, 1} | accb | có |
| 3 | {3, 1, 0} | ccb | **không** |

⇒ đáp án `2`.

**Quan sát:** cột cuối có dạng `có, có, …, có, không, …, không` — vì tập chỉ số bị xóa **chỉ lớn dần** khi `k` tăng, và xóa thêm ký tự chỉ làm việc khớp `p` khó hơn. Dạng "đơn điệu" này cho phép tìm điểm chuyển bằng nhị phân.

In [5]:
s, p, rem = "abcacb", "ab", [3, 1, 0]
for k in range(len(rem) + 1):
    removed = set(rem[:k])
    t = "".join(ch for i, ch in enumerate(s) if i not in removed)
    print(f"k={k}: xóa {sorted(removed)} -> {t!r}, dãy con? {is_subsequence(p, t)}")

k=0: xóa [] -> 'abcacb', dãy con? True
k=1: xóa [3] -> 'abccb', dãy con? True
k=2: xóa [1, 3] -> 'accb', dãy con? True
k=3: xóa [0, 1, 3] -> 'ccb', dãy con? False


## 5. Cách 1 — Brute Force / baseline: thử `k` tăng dần

In [6]:
def max_removals_linear(s, p, removable):
    removed = [False] * len(s)
    k = 0
    for idx in removable:
        removed[idx] = True                           # thử xóa thêm một ký tự
        j = 0
        for i, ch in enumerate(s):
            if not removed[i] and j < len(p) and ch == p[j]:
                j += 1
        if j < len(p):
            return k                                  # xóa thêm ký tự này thì hỏng
        k += 1
    return k

for (s, p, rem), want in examples:
    print(max_removals_linear(s, p, rem), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** tới `r = |removable|` lần kiểm tra, mỗi lần `O(n)` ⇒ `O(r · n)` — `10¹⁰` worst. Space `O(n)`.

**Nút thắt:** thử từng `k`; tính đơn điệu cho phép nhảy.

In [7]:
for n in [3000, 8000]:
    s = "a" * n; p = "a"; rem = list(range(n - 1))          # mọi lần xóa đều còn "a" -> phải thử hết
    t0 = time.perf_counter(); r = max_removals_linear(s, p, rem)
    print(f"n={n}: {r}, {time.perf_counter() - t0:.3f}s")

n=3000: 2999, 0.291s


n=8000: 7999, 1.593s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đơn điệu):** gọi `ok(k)` = "`p` là dãy con của `s` sau khi xóa `removable[:k]`". Nếu `ok(k)` đúng thì `ok(k')` đúng với mọi `k' < k`: chuỗi sau `k'` lần xóa là **siêu dãy** của chuỗi sau `k` lần xóa (có thêm ký tự), mà `p` là dãy con của chuỗi nhỏ hơn ⇒ cũng là dãy con của chuỗi lớn hơn.

⇒ `ok` có dạng `T, T, …, T, F, …, F` ⇒ **nhị phân** tìm `k` lớn nhất có `ok(k)`; `ok(0)` luôn đúng.

**Insight 2 (thời điểm xóa):** `when[i] = t` nếu `removable[t] = i`, ngược lại `∞`. Sau `k` lần xóa, ký tự `i` còn ⇔ `when[i] ≥ k`. Mỗi lần kiểm tra `ok(k)` chỉ cần so sánh số nguyên, không dựng `set`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhị phân + tập chỉ số bị xóa

In [8]:
def max_removals_binary_set(s, p, removable):
    def ok(k):
        removed = set(removable[:k])
        j = 0
        for i, ch in enumerate(s):
            if i not in removed and j < len(p) and ch == p[j]:
                j += 1
        return j == len(p)

    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for (s, p, rem), want in examples:
    print(max_removals_binary_set(s, p, rem), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** `log r` lần kiểm tra, mỗi lần `O(n + k)` ⇒ `O((n + r) log r)`. Space `O(r)`.

### 7.2 Cách 3 — Nhị phân + mảng thời điểm xóa

In [9]:
def max_removals_when(s, p, removable):
    INF = len(removable)
    when = [INF] * len(s)
    for t, idx in enumerate(removable):
        when[idx] = t
    m = len(p)

    def ok(k):
        j = 0
        for i, ch in enumerate(s):
            if when[i] >= k and ch == p[j]:
                j += 1
                if j == m:
                    return True
        return False

    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for (s, p, rem), want in examples:
    print(max_removals_when(s, p, rem), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


Đặt `INF = len(removable)` (lớn hơn mọi `k` hợp lệ) để ký tự không bao giờ bị xóa luôn thỏa `when[i] ≥ k`. Dừng sớm khi `j == m` (đã khớp hết `p`).

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** Time `O(n log r)`, Space `O(n)`.

### 7.3 Cách 4 — Kiểm tra bằng iterator trên chuỗi đã lọc

Dựng chuỗi còn lại bằng `join` (vòng C) rồi kiểm tra bằng iterator — mỗi lần kiểm tra vẫn `O(n)` nhưng hằng số nhỏ hơn vòng `for` thuần Python.

In [10]:
def max_removals_iter(s, p, removable):
    INF = len(removable)
    when = [INF] * len(s)
    for t, idx in enumerate(removable):
        when[idx] = t

    def ok(k):
        kept = "".join(ch for ch, w in zip(s, when) if w >= k)
        it = iter(kept)
        return all(c in it for c in p)

    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for (s, p, rem), want in examples:
    print(max_removals_iter(s, p, rem), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** như Cách 3.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `s = "abcbddddd"`, `p = "abcd"`, `removable = [3,2,1,4,5,6]`. Mảng `when`:

In [11]:
def trace_when(s, p, removable):
    INF = len(removable); when = [INF] * len(s)
    for t, idx in enumerate(removable): when[idx] = t
    print("chỉ số :", list(range(len(s))))
    print("ký tự  :", list(s))
    print("when   :", [w if w < INF else "∞" for w in when])
    def ok(k):
        kept = "".join(ch for ch, w in zip(s, when) if w >= k)
        res = is_subsequence(p, kept)
        return res, kept
    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2; res, kept = ok(mid)
        print(f"[lo={lo}, hi={hi}] thử k={mid}: còn lại {kept!r} -> {'đủ, lo = mid' if res else 'hỏng, hi = mid - 1'}")
        if res: lo = mid
        else: hi = mid - 1
    print("đáp án:", lo)
    return lo

trace_when("abcbddddd", "abcd", [3, 2, 1, 4, 5, 6])

chỉ số : [0, 1, 2, 3, 4, 5, 6, 7, 8]
ký tự  : ['a', 'b', 'c', 'b', 'd', 'd', 'd', 'd', 'd']
when   : ['∞', 2, 1, 0, 3, 4, 5, '∞', '∞']
[lo=0, hi=6] thử k=3: còn lại 'addddd' -> hỏng, hi = mid - 1
[lo=0, hi=2] thử k=1: còn lại 'abcddddd' -> đủ, lo = mid
[lo=1, hi=2] thử k=2: còn lại 'abddddd' -> hỏng, hi = mid - 1
đáp án: 1


1

In [12]:
trace_when("abcacb", "ab", [3, 1, 0])

chỉ số : [0, 1, 2, 3, 4, 5]
ký tự  : ['a', 'b', 'c', 'a', 'c', 'b']
when   : [2, 1, '∞', 0, '∞', '∞']
[lo=0, hi=3] thử k=2: còn lại 'accb' -> đủ, lo = mid
[lo=2, hi=3] thử k=3: còn lại 'ccb' -> hỏng, hi = mid - 1
đáp án: 2


2

Bảng tóm tắt trace đầu:

| lo | hi | mid | chuỗi còn lại | kết luận |
|---|---|---|---|---|
| 0 | 6 | 3 | a d d d d d (xóa 3, 2, 1) | hỏng ⇒ hi = 2 |
| 0 | 2 | 1 | abc d d d d d (xóa 3) | đủ ⇒ lo = 1 |
| 1 | 2 | 2 | ab d d d d d (xóa 3, 2) | hỏng ⇒ hi = 1 |
| 1 | 1 | — | — | **1** |

(Chuỗi chính xác xem output.)

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def maximumRemovals(self, s: str, p: str, removable: List[int]) -> int:
        INF = len(removable)
        when = [INF] * len(s)
        for t, idx in enumerate(removable):
            when[idx] = t
        m = len(p)

        def ok(k: int) -> bool:
            j = 0
            for i, ch in enumerate(s):
                if when[i] >= k and ch == p[j]:
                    j += 1
                    if j == m:
                        return True
            return False

        lo, hi = 0, len(removable)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

sol = Solution()
for (s, p, rem), want in examples:
    got = sol.maximumRemovals(s, p, rem)
    print(got, "OK" if got == want else "SAI")

2 OK
1 OK
0 OK


## 10. Vì sao thuật toán đúng

**Đơn điệu.** Gọi `S_k` là chuỗi còn lại sau khi xóa `removable[:k]`. Vì `removable[:k] ⊆ removable[:k+1]`, `S_{k+1}` là dãy con của `S_k`. Nếu `p` là dãy con của `S_{k+1}` thì (bắc cầu) `p` là dãy con của `S_k`. Vậy `ok(k+1) ⇒ ok(k)`, tức `ok` có dạng tiền tố đúng.

**Kiểm tra dãy con tham lam đúng.** Nếu `p` là dãy con của `t`, khớp `p[j]` với lần xuất hiện **sớm nhất** có thể của nó trong `t` không làm mất khả năng khớp phần còn lại (lập luận đổi chỗ: mọi phép nhúng có thể "kéo" về sớm hơn).

**`when`.** Ký tự `i` bị xóa trong `S_k` ⇔ `i ∈ removable[:k]` ⇔ `when[i] < k`. ✓

**Nhị phân.** Invariant: `ok(lo)` đúng (ban đầu `ok(0)` theo đề), mọi `k > hi` sai. `mid = ⌈(lo + hi)/2⌉ > lo` nên khoảng luôn co lại. Kết thúc `lo = hi` = `k` lớn nhất thỏa.

## 11. Phân tích Time / Space

Gọi `n = |s| ≤ 10⁵`, `r = |removable| < n`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử `k` tăng dần | `O(r · n)` | `O(n)` | kiểm chứng |
| 2. Nhị phân + `set` | `O((n + r) log r)` | `O(r)` | |
| 3. Nhị phân + `when` | `O(n log r)` | `O(n)` | **nộp bài** |
| 4. + iterator | `O(n log r)` | `O(n)` | hằng số nhỏ |

In [14]:
random.seed(1)
n = 10**5
s_big = "".join(random.choice("ab") for _ in range(n))
p_big = "".join(random.choice("ab") for _ in range(n // 10))
# bảo đảm p là dãy con: lấy p là dãy con ngẫu nhiên của s
idxs = sorted(random.sample(range(n), n // 10)); p_big = "".join(s_big[i] for i in idxs)
rem_big = random.sample(range(n), n - 1)
for name, fn in [("binary_set", max_removals_binary_set), ("when", max_removals_when), ("iter", max_removals_iter),
                 ("submit", Solution().maximumRemovals)]:
    t0 = time.perf_counter(); r = fn(s_big, p_big, rem_big)
    print(f"{name:10s}: {r}  {time.perf_counter() - t0:.3f}s")

binary_set: 79997  0.131s


when      : 79997  0.088s
iter      : 79997  0.065s


submit    : 79997  0.068s


## 12. Bộ test

Sinh input đúng ràng buộc: `p` là dãy con ngẫu nhiên của `s`, `removable` là tập chỉ số phân biệt (thứ tự ngẫu nhiên, độ dài `< |s|`).

In [15]:
def gen(rng, n_max=12, alpha="abc"):
    n = rng.randint(1, n_max)
    s = "".join(rng.choice(alpha) for _ in range(n))
    m = rng.randint(1, n)
    p = "".join(s[i] for i in sorted(rng.sample(range(n), m)))
    rem = rng.sample(range(n), rng.randint(0, n - 1))
    return s, p, rem

fns = [max_removals_binary_set, max_removals_when, max_removals_iter, lambda a, b, c: Solution().maximumRemovals(a, b, c)]

for (s, p, rem), want in examples:
    assert max_removals_linear(s, p, rem) == want
    for fn in fns:
        assert fn(s, p, rem) == want

edge = [
    (("a", "a", []), 0),
    (("ab", "a", [1]), 1),
    (("ab", "a", [0]), 0),
    (("aaaa", "aa", [0, 1, 2]), 2),
    (("abc", "abc", [2, 1]), 0),
]
for (s, p, rem), want in edge:
    assert max_removals_linear(s, p, rem) == want, (s, p, rem)
    for fn in fns:
        assert fn(s, p, rem) == want, (fn, s, p, rem)

rng = random.Random(2024)
for _ in range(2000):
    s, p, rem = gen(rng, alpha=rng.choice(["a", "ab", "abc"]))
    want = max_removals_linear(s, p, rem)
    for fn in fns:
        assert fn(s, p, rem) == want, (fn, s, p, rem)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp thử tuyến tính")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp thử tuyến tính


In [16]:
# kiểm tra định nghĩa: ok(ans) đúng và (nếu ans < r) ok(ans + 1) sai
def ok_def(s, p, rem, k):
    removed = set(rem[:k])
    return is_subsequence(p, "".join(ch for i, ch in enumerate(s) if i not in removed))

rng = random.Random(7)
for _ in range(500):
    s, p, rem = gen(rng, n_max=40)
    a = Solution().maximumRemovals(s, p, rem)
    assert ok_def(s, p, rem, a) and (a == len(rem) or not ok_def(s, p, rem, a + 1))
print("đáp án thỏa định nghĩa 'lớn nhất' (500 bộ)")

đáp án thỏa định nghĩa 'lớn nhất' (500 bộ)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Nhị phân làm tròn xuống (`mid = (lo + hi) // 2`) với `lo = mid` ⇒ kẹt

In [17]:
def wrong_round_down(s, p, removable, max_iter=60):
    lo, hi = 0, len(removable); it = 0
    while lo < hi and it < max_iter:
        mid = (lo + hi) // 2                           # SAI: làm tròn xuống
        if ok_def(s, p, removable, mid): lo = mid
        else: hi = mid - 1
        it += 1
    return lo, it

print(wrong_round_down("abcacb", "ab", [3, 1, 0]), "(vòng lặp chạm giới hạn 60 -> kẹt)")

(2, 60) (vòng lặp chạm giới hạn 60 -> kẹt)


**Sửa:** `mid = (lo + hi + 1) // 2`.

### Lỗi 2 — Xóa **ký tự** (theo giá trị) thay vì **vị trí**

In [18]:
def wrong_remove_by_value(s, p, removable):
    best = 0
    for k in range(len(removable) + 1):
        bad = {s[i] for i in removable[:k]}             # SAI: xóa mọi ký tự cùng giá trị
        t = "".join(ch for ch in s if ch not in bad)
        if is_subsequence(p, t): best = k
        else: break
    return best

for (s, p, rem), want in examples:
    print("sai =", wrong_remove_by_value(s, p, rem), "| đúng =", want)

sai = 0 | đúng = 2
sai = 0 | đúng = 1
sai = 0 | đúng = 0


**Sửa:** chỉ xóa đúng các **chỉ số** trong `removable[:k]`.

### Lỗi 3 — Dùng `removable[k]` (lệch một) khi xác định tập xóa

In [19]:
def wrong_off_by_one(s, p, removable):
    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        removed = set(removable[:mid + 1])             # SAI: xóa mid + 1 ký tự
        t = "".join(ch for i, ch in enumerate(s) if i not in removed)
        if is_subsequence(p, t): lo = mid
        else: hi = mid - 1
    return lo

for (s, p, rem), want in examples:
    print("sai =", wrong_off_by_one(s, p, rem), "| đúng =", want)

sai = 1 | đúng = 2
sai = 0 | đúng = 1
sai = 0 | đúng = 0


**Sửa:** `k` lần xóa ứng với `removable[:k]` (các chỉ số `0..k−1`).

### Lỗi 4 — Cận trên `hi = len(removable) − 1` (bỏ sót trường hợp xóa hết được)

In [20]:
def wrong_hi(s, p, removable):
    lo, hi = 0, max(0, len(removable) - 1)            # SAI: k có thể bằng len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok_def(s, p, removable, mid): lo = mid
        else: hi = mid - 1
    return lo

print("sai =", wrong_hi("abcd", "a", [1, 2, 3]), "| đúng =", Solution().maximumRemovals("abcd", "a", [1, 2, 3]))

sai = 2 | đúng = 3


**Sửa:** `hi = len(removable)` (đề cho `0 ≤ k ≤ removable.length`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm `k` lớn nhất sao cho sau `k` bước (tích lũy) điều kiện còn đúng" + điều kiện **đơn điệu** ⇒ nhị phân trên `k`.
- Dãy con ⇒ hai con trỏ tham lam (hoặc iterator).
- "Thời điểm bị xóa" `when[i]` biến "tập thay đổi theo `k`" thành so sánh số nguyên — kỹ thuật dùng lại được ở nhiều bài "xóa/thêm theo thứ tự rồi hỏi".
- Họ bài: 392 *Is Subsequence*, 1482 *Minimum Days to Make m Bouquets*, 2009, 1552 *Magnetic Force Between Two Balls* (nhị phân trên đáp án).

## 15. Biến thể của bài

1. **Trả về chuỗi `s` sau khi xóa tối đa** — dùng `when[i] ≥ ans`.
2. **Thứ tự xóa tùy chọn** (được chọn ký tự nào xóa): xóa tối đa `|s| − |p|` ký tự (giữ đúng một phép nhúng của `p`).
3. **Nhiều chuỗi `p₁, …, p_q`** cùng phải còn là dãy con: `ok(k)` = AND của các kiểm tra — vẫn đơn điệu.
4. **Đếm số `k` hợp lệ** = `ans + 1`.

In [21]:
def removed_string(s, p, removable):
    """Biến thể 1."""
    ans = Solution().maximumRemovals(s, p, removable)
    gone = set(removable[:ans])
    return ans, "".join(ch for i, ch in enumerate(s) if i not in gone)

def max_removals_multi(s, ps, removable):
    """Biến thể 3: mọi p trong ps phải còn là dãy con."""
    INF = len(removable); when = [INF] * len(s)
    for t, idx in enumerate(removable): when[idx] = t
    def ok(k):
        kept = "".join(ch for ch, w in zip(s, when) if w >= k)
        return all(is_subsequence(p, kept) for p in ps)
    lo, hi = 0, len(removable)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    return lo

print(removed_string("abcacb", "ab", [3, 1, 0]))
print(max_removals_multi("abcacb", ["ab", "cb"], [3, 1, 0]), max_removals_multi("abcacb", ["ab", "ac"], [3, 1, 0]))

(2, 'accb')
2 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử tăng dần | kiểm tra từng `k` | `O(r n)` | `O(n)` | kiểm chứng |
| 2. Nhị phân + `set` | đơn điệu | `O((n + r) log r)` | `O(r)` | dễ hiểu |
| 3. Nhị phân + `when` | so sánh số nguyên | `O(n log r)` | `O(n)` | **nộp bài** |
| 4. + iterator | lọc bằng `join` | `O(n log r)` | `O(n)` | hằng số nhỏ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumRemovals("qobftgcueho", "obue", [5,3,0,6,4,9,10,7,2,8])`?

**Bài 2 (tính tay):** lập bảng `when` và chạy nhị phân bằng tay cho `s = "aabbaa"`, `p = "aba"`, `removable = [4, 0, 2, 5]`.

**Bài 3 (điền TODO):** hoàn thiện `ok_when(s, p, when, k)` kiểm tra `p` còn là dãy con sau `k` lần xóa, dùng `when`.

**Bài 4 (tự viết lại):** viết lời giải nhị phân theo khuôn "tìm `k` **nhỏ nhất** làm hỏng" (`first_bad`), trả về `first_bad − 1`.

**Bài 5 (mini challenge):** biến thể 2 — được **tự chọn** thứ tự xóa trong tập `removable` (không bắt buộc theo thứ tự cho trước): tối đa số chỉ số trong `removable` có thể xóa. Gợi ý: cần chọn một phép nhúng của `p` vào `s` dùng **ít** chỉ số thuộc `removable` nhất — DP `O(|s| · |p|)`. Kiểm chứng bằng vét cạn tập con. (Biến thể tự đặt.)

In [22]:
def ok_when(s, p, when, k):
    # TODO: hai con trỏ, chỉ xét ký tự i có when[i] >= k
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Đây là một ví dụ phổ biến của bài. (2) Chạy `trace_when`. (3) Đáp án `7` (xóa được 7 chỉ số đầu mà "obue" vẫn là dãy con).

**Bài 2:** (1) `when = [1, ∞, 2, ∞, 0, 3]`. (2) `k = 2`: còn chỉ số `{1, 2, 3, 5}` ⇒ `"abba"` có `"aba"`. `k = 3`: còn `{1, 3, 5}` ⇒ `"aba"` ✓. (3) `k = 4`: còn `{1, 3}` ⇒ `"ab"` ✗ ⇒ đáp án `3`.

**Bài 3:** (1) `j = 0`. (2) `if when[i] >= k and j < len(p) and s[i] == p[j]: j += 1`. (3) `return j == len(p)`.

**Bài 4:** (1) `first_bad` trên `[1, r + 1]` (giá trị `r + 1` = "không bao giờ hỏng"). (2) `mid = (lo + hi) // 2`; hỏng ⇒ `hi = mid`. (3) Trả `lo − 1`.

**Bài 5:** (1) `dp[i][j]` = số chỉ số thuộc `removable` ít nhất phải **giữ lại** để nhúng `p[:j]` vào `s[:i]`. (2) Chuyển: bỏ qua `s[i−1]` hoặc khớp `s[i−1] = p[j−1]` (cộng 1 nếu `i−1 ∈ removable`). (3) Đáp án `|removable| − dp[n][m]`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
print("Bài 1:", Solution().maximumRemovals("qobftgcueho", "obue", [5, 3, 0, 6, 4, 9, 10, 7, 2, 8]))
assert Solution().maximumRemovals("qobftgcueho", "obue", [5, 3, 0, 6, 4, 9, 10, 7, 2, 8]) == \
       max_removals_linear("qobftgcueho", "obue", [5, 3, 0, 6, 4, 9, 10, 7, 2, 8])

# Bài 2
assert Solution().maximumRemovals("aabbaa", "aba", [4, 0, 2, 5]) == 3

# Bài 3
def ok_when_ans(s, p, when, k):
    j = 0
    for i, ch in enumerate(s):
        if when[i] >= k and j < len(p) and ch == p[j]:
            j += 1
    return j == len(p)

# Bài 4
def max_removals_first_bad(s, p, removable):
    INF = len(removable); when = [INF] * len(s)
    for t, idx in enumerate(removable): when[idx] = t
    lo, hi = 1, len(removable) + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if not ok_when_ans(s, p, when, mid): hi = mid
        else: lo = mid + 1
    return lo - 1

rng = random.Random(231)
for _ in range(1500):
    s, p, rem = gen(rng)
    assert max_removals_first_bad(s, p, rem) == max_removals_linear(s, p, rem)
print("Bài 1-4: OK")

Bài 1: 7
Bài 1-4: OK


In [24]:
# Bài 5 — tự chọn ký tự để xóa trong tập removable
from itertools import combinations

def max_removals_free(s, p, removable):
    n, m = len(s), len(p); R = set(removable); INF = float("inf")
    dp = [[INF] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j]                                 # không dùng s[i-1]
            if s[i - 1] == p[j - 1] and dp[i - 1][j - 1] + (i - 1 in R) < dp[i][j]:
                dp[i][j] = dp[i - 1][j - 1] + (i - 1 in R)          # khớp, phải giữ s[i-1]
    return len(removable) - dp[n][m]

def max_removals_free_brute(s, p, removable):
    for size in range(len(removable), -1, -1):
        for sub in combinations(removable, size):
            gone = set(sub)
            if is_subsequence(p, "".join(ch for i, ch in enumerate(s) if i not in gone)):
                return size
    return 0

rng = random.Random(232)
for _ in range(400):
    s, p, rem = gen(rng, n_max=9)
    a = max_removals_free(s, p, rem)
    assert a == max_removals_free_brute(s, p, rem), (s, p, rem)
    assert a >= Solution().maximumRemovals(s, p, rem)        # tự chọn thì không tệ hơn
print("Bài 5: DP khớp vét cạn; ví dụ 1 tự chọn ->", max_removals_free("abcacb", "ab", [3, 1, 0]))

Bài 5: DP khớp vét cạn; ví dụ 1 tự chọn -> 2


## 20. Template nhớ nhanh + Checklist

```python
when = [len(removable)] * len(s)                 # thời điểm bị xóa (∞ = không bao giờ)
for t, idx in enumerate(removable): when[idx] = t
def ok(k):                                       # p còn là dãy con sau k lần xóa?
    j = 0
    for i, ch in enumerate(s):
        if when[i] >= k and ch == p[j]:
            j += 1
            if j == len(p): return True
    return False
lo, hi = 0, len(removable)
while lo < hi:
    mid = (lo + hi + 1) // 2                     # LỚN nhất thỏa -> làm tròn lên
    if ok(mid): lo = mid
    else: hi = mid - 1
return lo
```

**Checklist trước khi submit:**
- [ ] Xóa theo **chỉ số**, không theo giá trị ký tự.
- [ ] `k` lần xóa = `removable[:k]`.
- [ ] `hi = len(removable)`; `ok(0)` luôn đúng.
- [ ] `mid` làm tròn lên.
- [ ] Dừng sớm khi đã khớp hết `p` (và cẩn thận `p[j]` khi `j == len(p)`).